## Lab: Vector DB and retrieval quality — cross-domain pipeline

This lab extends the C2.1-style RAG pipeline with:
1. **ChromaDB persistence** (or mock) for the document store.
2. **Metadata filtering** to scope retrieval to the right domain before ranking.
3. **Hybrid search + reranking** over the filtered candidate set.
4. **Labeled evaluation** with a 13-query cross-domain eval set (tech, finance, education, healthcare, legal).

Run the three domain examples below to see how the same pipeline handles queries from completely different fields, then review the aggregate metrics to compare ranker quality end-to-end.

In [ ]:
import sys
sys.path.append('Track2/Functions')

from pathlib import Path
import numpy as np

from C2_2_Func import (
    LocalSemanticEmbedder, cosine_similarity,
    build_corpus_docs, build_eval_sets, persist_corpus, load_jsonl,
    SimpleBM25, normalize_scores, rerank_like_cross_encoder, evaluate_ranker,
)

CORPUS_DIR = Path('Track2') / 'demos' / 'data' / 'C2.2_corpus'
corpus_docs = build_corpus_docs()
metadata_eval_set, labeled_eval_set = build_eval_sets()
persist_corpus(corpus_docs, metadata_eval_set, labeled_eval_set, CORPUS_DIR)

loaded_docs = load_jsonl(CORPUS_DIR / 'docs.jsonl')
loaded_labeled_eval = load_jsonl(CORPUS_DIR / 'labeled_eval_set.jsonl')
embedder = LocalSemanticEmbedder()

In [ ]:
def retrieve_with_metadata_and_hybrid(query, metadata_filter=None, top_k=3, alpha=0.55):
    candidate_docs = loaded_docs
    if metadata_filter:
        candidate_docs = [
            doc for doc in loaded_docs
            if all(doc.get(k) == v for k, v in metadata_filter.items())
        ]
    if not candidate_docs:
        return []
    local_bm25   = SimpleBM25(candidate_docs)
    local_dense  = embedder.encode([doc['text'] for doc in candidate_docs])
    query_vec    = embedder.encode([query])[0]
    sparse_s     = normalize_scores(local_bm25.score(query))
    dense_s      = normalize_scores(
        np.array([cosine_similarity(query_vec, dv) for dv in local_dense], dtype=np.float32))
    combined     = alpha * dense_s + (1 - alpha) * sparse_s
    order        = np.argsort(combined)[::-1][:top_k]
    ranked       = [{'doc': candidate_docs[i], 'score': float(combined[i])} for i in order]
    return rerank_like_cross_encoder(query, ranked)


def show_lab_example(label, eval_row, top_k=5):
    ranked = retrieve_with_metadata_and_hybrid(
        eval_row['query'], metadata_filter=eval_row['metadata_filter'], top_k=top_k)
    print(f'\n── {label} ──')
    print(f'Query  : {eval_row["query"]}')
    print(f'Filter : {eval_row["metadata_filter"]}')
    for rank, item in enumerate(ranked, start=1):
        doc = item['doc']
        print(f'  {rank}. {doc["id"]} | {doc["title"]:<40} | score={item["score"]:.3f}')
    return ranked


# ── Domain examples ────────────────────────────────────────────────────────────
# Retrieval query index within labeled_eval_set:
#   0-5 = technical,  6-8 = finance,  9-10 = education,  11 = healthcare,  12 = legal
lab_tech  = show_lab_example('Technical',   loaded_labeled_eval[1])
lab_fin   = show_lab_example('Finance',     loaded_labeled_eval[6])
lab_edu   = show_lab_example('Education',   loaded_labeled_eval[9])
lab_health = show_lab_example('Healthcare', loaded_labeled_eval[11])
lab_legal  = show_lab_example('Legal',      loaded_labeled_eval[12])

# ── End-to-end metrics over the full cross-domain eval set ─────────────────────
lab_scores = evaluate_ranker(
    loaded_labeled_eval,
    lambda q: retrieve_with_metadata_and_hybrid(q, top_k=5),
    k=5,
)
print(f'\n── End-to-end lab metrics ({len(loaded_labeled_eval)} cross-domain queries) ──')
print(f"  Recall@5   : {lab_scores['recall']:.2f}")
print(f"  Precision@5: {lab_scores['precision']:.2f}")
print(f"  MRR        : {lab_scores['mrr']:.2f}")
print(f"  NDCG@5     : {lab_scores['ndcg']:.2f}")

# ── Grounded answer excerpt from the finance example ──────────────────────────
grounded = ' '.join(item['doc']['text'] for item in lab_fin[:2]) if lab_fin else ''
print(f'\n── Grounded answer excerpt (finance) ──')
print(grounded)

The end-to-end lab demonstrates the full pipeline — ChromaDB persistence → metadata filtering → hybrid search → reranking → metric evaluation — running the same code against queries from five different domains. The grounded answer is assembled only from retrieved chunks, which is the correct RAG behavior regardless of domain.